# Full Vehicle Simulation
## VehicleBody 통합 시뮬레이션

차체 6-DOF 동역학 + 4개 E-Corner 통합

## 1. 모듈 임포트 및 초기화

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# 프로젝트 루트 경로 추가
notebook_dir = Path.cwd()
project_root = notebook_dir.parent.parent.parent.parent
sys.path.insert(0, str(project_root))

from vehicle_sim.models.vehicle_body.vehicle_body import VehicleBody, VehicleBodyParameters
from vehicle_sim.utils.config_loader import load_param

print(f"프로젝트 루트: {project_root}")
print("모듈 로드 완료")

프로젝트 루트: /home/a/Downloads/SeohanModel_ver7/SeohanModel
모듈 로드 완료


## 2. 차량 파라미터 로드

In [2]:
# YAML 파라미터 로드
config_path = project_root / "vehicle_sim" / "models" / "params" / "vehicle_standard.yaml"

body_param = load_param('vehicle_body', str(config_path))
geom_param = load_param('vehicle_spec', str(config_path))['geometry']
susp_param = load_param('suspension', str(config_path))
physics_param = load_param('physics', str(config_path))

# VehicleBodyParameters 생성
params = VehicleBodyParameters(
    m=float(body_param['m']),
    Ixx=float(body_param['inertia']['Ixx']),
    Iyy=float(body_param['inertia']['Iyy']),
    Izz=float(body_param['inertia']['Izz']),
    Ixz=float(body_param['inertia']['Ixz']),
    h_CG=float(susp_param['z_CG0']),
    a=float(geom_param['L_wheelbase']) / 2.0,
    b=float(geom_param['L_wheelbase']) / 2.0,
    L_track=float(geom_param['L_track']),
    L_wheelbase=float(geom_param['L_wheelbase']),
    g=float(physics_param['g'])
)

print("차체 파라미터:")
print(f"  질량: {params.m} kg")
print(f"  관성: Ixx={params.Ixx}, Iyy={params.Iyy}, Izz={params.Izz} kg·m²")
print(f"  CG 높이: {params.h_CG} m")
print(f"  휠베이스: {params.L_wheelbase} m, 트랙: {params.L_track} m")

차체 파라미터:
  질량: 1300.0 kg
  관성: Ixx=700.0, Iyy=3500.0, Izz=3500.0 kg·m²
  CG 높이: 0.967 m
  휠베이스: 2.8 m, 트랙: 1.6 m


## 3. VehicleBody 초기화

In [3]:
# VehicleBody 생성 (config_path를 E-Corner 초기화에 전달)
vehicle = VehicleBody(parameters=params, config_path=str(config_path))

# 초기 상태 설정
vehicle.reset()
initial_speed = 5.0  # 초기 속도 5 m/s (18 km/h) - 더 천천히
vehicle.state.velocity_x = initial_speed

# 초기 휠 속도도 차체 속도에 맞춰 초기화 (중요!)
for label in vehicle.wheel_labels:
    corner = vehicle.corners[label]
    # 휠 반지름 가져오기
    R_wheel = corner.drive.params.R_wheel
    # 초기 휠 각속도 = velocity_x / R_wheel
    corner.drive.state.wheel_speed = initial_speed / R_wheel
    print(f"  {label} 초기 휠 속도: {corner.drive.state.wheel_speed:.2f} rad/s (v={initial_speed:.1f} m/s)")

print("\n초기 상태:")
print(f"  위치: x={vehicle.state.x:.3f}, y={vehicle.state.y:.3f}, heave={vehicle.state.heave:.3f} m")
print(f"  속도: velocity_x={vehicle.state.velocity_x:.3f} m/s ({vehicle.state.velocity_x*3.6:.1f} km/h)")
print(f"  코너: {vehicle.wheel_labels}")

# 평형 체크: 각 코너의 F_s 확인
print("\n초기 평형 체크:")
total_Fs = 0.0
for label in vehicle.wheel_labels:
    corner_state = vehicle.corners[label].get_state()
    F_s = corner_state['F_s']
    total_Fs += F_s
    print(f"  {label}: F_s = {F_s:.2f} N")

m_total = params.m + 4 * 50.0  # sprung + 4 * unsprung
weight_total = m_total * params.g
print(f"\n  Σ F_s = {total_Fs:.2f} N")
print(f"  Total weight (mg) = {weight_total:.2f} N")
print(f"  Difference = {total_Fs - weight_total:.2f} N ({(total_Fs - weight_total)/weight_total*100:.2f}%)")

  FL 초기 휠 속도: 15.82 rad/s (v=5.0 m/s)
  FR 초기 휠 속도: 15.82 rad/s (v=5.0 m/s)
  RR 초기 휠 속도: 15.82 rad/s (v=5.0 m/s)
  RL 초기 휠 속도: 15.82 rad/s (v=5.0 m/s)

초기 상태:
  위치: x=0.000, y=0.000, heave=0.000 m
  속도: velocity_x=5.000 m/s (18.0 km/h)
  코너: ['FL', 'FR', 'RR', 'RL']

초기 평형 체크:
  FL: F_s = 3188.25 N
  FR: F_s = 3188.25 N
  RR: F_s = 3188.25 N
  RL: F_s = 3188.25 N

  Σ F_s = 12753.00 N
  Total weight (mg) = 14715.00 N
  Difference = -1962.00 N (-13.33%)


## 4. 시뮬레이션 시나리오 설정

### 시나리오: 노면 범프 통과
- 10 m/s로 주행 중 5m 지점에서 사인 범프 통과
- 범프 길이: 1m, 높이: 5cm

In [ ]:
# 시뮬레이션 파라미터
dt = 0.001  # 1ms
t_end = 20.0  # 3초
n_steps = int(t_end / dt)
t = np.linspace(0, t_end, n_steps)

# 코너별 노면 프로파일 (각각 다른 특성)
def road_profile_FL(x_pos):
    """FL 노면: 3mm 사인 범프"""
    bump_start = 0.5  # 0.5m에서 시작 (가까이)
    bump_length = 2.5
    bump_height = 0.03  # 3mm
    if bump_start <= x_pos <= bump_start + bump_length:
        phase = (x_pos - bump_start) / bump_length * np.pi
        return bump_height * np.sin(phase)
    return 0.0

def road_profile_FR(x_pos):
    """FR 노면: 5mm 사인 범프 (FL보다 높음)"""
    bump_start = 0.7  # FL보다 0.2m 뒤
    bump_length = 2.5
    bump_height = 0.05  # 5mm
    if bump_start <= x_pos <= bump_start + bump_length:
        phase = (x_pos - bump_start) / bump_length * np.pi
        return bump_height * np.sin(phase)
    return 0.0

def road_profile_RL(x_pos):
    """RL 노면: 4mm 사인 범프"""
    bump_start = 1.9  # 전륜보다 1.4m 뒤 (휠베이스)
    bump_length = 2.5
    bump_height = 0.04  # 4mm
    if bump_start <= x_pos <= bump_start + bump_length:
        phase = (x_pos - bump_start) / bump_length * np.pi
        return bump_height * np.sin(phase)
    return 0.0

def road_profile_RR(x_pos):
    """RR 노면: 6mm 사인 범프 (가장 높음)"""
    bump_start = 2.1  # RL보다 0.2m 뒤
    bump_length = 2.5
    bump_height = 0.06  # 6mm
    if bump_start <= x_pos <= bump_start + bump_length:
        phase = (x_pos - bump_start) / bump_length * np.pi
        return bump_height * np.sin(phase)
    return 0.0

road_profiles = {
    'FL': road_profile_FL,
    'FR': road_profile_FR,
    'RL': road_profile_RL,
    'RR': road_profile_RR
}

print(f"시뮬레이션 설정:")
print(f"  dt: {dt*1000:.1f} ms")
print(f"  총 시간: {t_end} s")
print(f"  스텝 수: {n_steps}")


시뮬레이션 설정:
  dt: 1.0 ms
  총 시간: 200.0 s
  스텝 수: 200000


## 5. 시뮬레이션 실행

In [ ]:
# 입력/출력 저장용 배열
input_traces = {
    'z_road_FL': np.zeros(n_steps),
    'z_road_FR': np.zeros(n_steps),
    'z_road_RL': np.zeros(n_steps),
    'z_road_RR': np.zeros(n_steps),
    'T_steer': np.zeros(n_steps),
    'T_brk': np.zeros(n_steps),
    'T_Drv': np.zeros(n_steps),
    'T_susp': np.zeros(n_steps)
}

output_traces = {
    'x': np.zeros(n_steps),
    'y': np.zeros(n_steps),
    'heave': np.zeros(n_steps),
    'roll': np.zeros(n_steps),
    'pitch': np.zeros(n_steps),
    'yaw': np.zeros(n_steps),
    'velocity_x': np.zeros(n_steps),
    'velocity_y': np.zeros(n_steps),
    'heave_dot': np.zeros(n_steps),
    'roll_rate': np.zeros(n_steps),
    'pitch_rate': np.zeros(n_steps),
    'yaw_rate': np.zeros(n_steps),
    # 코너별 힘 추가
    'F_s_FL': np.zeros(n_steps),
    'F_s_FR': np.zeros(n_steps),
    'F_s_RL': np.zeros(n_steps),
    'F_s_RR': np.zeros(n_steps),
    'F_x_FL': np.zeros(n_steps),
    'F_x_FR': np.zeros(n_steps),
    'F_x_RL': np.zeros(n_steps),
    'F_x_RR': np.zeros(n_steps),
    'F_y_FL': np.zeros(n_steps),
    'F_y_FR': np.zeros(n_steps),
    'F_y_RL': np.zeros(n_steps),
    'F_y_RR': np.zeros(n_steps),
    'F_z_FL': np.zeros(n_steps),
    'F_z_FR': np.zeros(n_steps),
    'F_z_RL': np.zeros(n_steps),
    'F_z_RR': np.zeros(n_steps),
    # 디버깅용 추가 변수
    'delta_s_FL': np.zeros(n_steps),
    'delta_s_FR': np.zeros(n_steps),
    'delta_s_RL': np.zeros(n_steps),
    'delta_s_RR': np.zeros(n_steps),
    'delta_t_FL': np.zeros(n_steps),
    'delta_t_FR': np.zeros(n_steps),
    'delta_t_RL': np.zeros(n_steps),
    'delta_t_RR': np.zeros(n_steps),
    'z_u_FL': np.zeros(n_steps),
    'z_u_FR': np.zeros(n_steps),
    'z_u_RL': np.zeros(n_steps),
    'z_u_RR': np.zeros(n_steps),
}

# 디버그/입력 시각화용 트레이스 (steering/slip)
debug_traces = {}
for label in vehicle.wheel_labels:
    for name in ['V_wheel_x', 'V_wheel_y', 'steering_angle', 'alpha', 'kappa', 'omega_wheel', 'M_align']:
        debug_traces[f'{name}_{label}'] = np.zeros(n_steps)

# 고정 드라이브 토크
T_Drv_fixed = 50.0  # [Nm] - 속도 유지용

print("시뮬레이션 시작...")
print(f"\n초기 코너 상태 (디버깅):")
for label in vehicle.wheel_labels:
    corner_state = vehicle.corners[label].suspension.get_state()
    print(f"  {label}: z_u_abs={corner_state['z_u_abs']:.4f}m, delta_s={corner_state['delta_s']*1000:.2f}mm, delta_t={corner_state['delta_t']*1000:.2f}mm")

for i in range(n_steps):
    # 1. 각 코너별 노면 높이 계산 (각각 다른 프로파일)
    z_road = {}
    for idx, label in enumerate(vehicle.wheel_labels):
        wheel_pos = vehicle.get_wheel_position(idx)
        z_road[label] = road_profiles[label](wheel_pos[0])
    
    # 1.5 업데이트 전 휠 속도 저장 (슬립각/슬립비 계산용)
    corner_body_inputs = vehicle.get_corner_inputs()
    for label in vehicle.wheel_labels:
        vx, vy = corner_body_inputs['wheel_velocities'][label]
        debug_traces[f'V_wheel_x_{label}'][i] = vx
        debug_traces[f'V_wheel_y_{label}'][i] = vy

    # 2. 입력 준비
    corner_inputs = {
        label: {
            "T_steer": 40.0, #4.0,
            "T_brk": 0.0,
            "T_Drv": T_Drv_fixed,
            "T_susp": 0.0,
            "z_road": 0.0 #z_road[label]
        }
        for label in vehicle.wheel_labels
    }
    
    # 3. 입력 저장
    input_traces['z_road_FL'][i] = z_road['FL']
    input_traces['z_road_FR'][i] = z_road['FR']
    input_traces['z_road_RL'][i] = z_road['RL']
    input_traces['z_road_RR'][i] = z_road['RR']
    input_traces['T_steer'][i] = 0.0
    input_traces['T_brk'][i] = 0.0
    input_traces['T_Drv'][i] = T_Drv_fixed
    input_traces['T_susp'][i] = 0.0
    
    # 4. VehicleBody 업데이트 (통합)
    vehicle.update(dt, corner_inputs)
    
    # 5. 출력 저장
    output_traces['x'][i] = vehicle.state.x
    output_traces['y'][i] = vehicle.state.y
    output_traces['heave'][i] = vehicle.state.heave
    output_traces['roll'][i] = vehicle.state.roll
    output_traces['pitch'][i] = vehicle.state.pitch
    output_traces['yaw'][i] = vehicle.state.yaw
    output_traces['velocity_x'][i] = vehicle.state.velocity_x
    output_traces['velocity_y'][i] = vehicle.state.velocity_y
    output_traces['heave_dot'][i] = vehicle.state.heave_dot
    output_traces['roll_rate'][i] = vehicle.state.roll_rate
    output_traces['pitch_rate'][i] = vehicle.state.pitch_rate
    output_traces['yaw_rate'][i] = vehicle.state.yaw_rate
    
    # 6. 코너별 힘 및 상태 저장
    for label in vehicle.wheel_labels:
        corner_state = vehicle.corners[label].get_state()
        sus_state = vehicle.corners[label].suspension.get_state()


        # Steering / slip 디버그값 저장
        steering_angle = corner_state['steering_angle']
        omega_wheel = corner_state['omega_wheel']

        debug_traces[f'steering_angle_{label}'][i] = steering_angle
        debug_traces[f'omega_wheel_{label}'][i] = omega_wheel
        debug_traces[f'M_align_{label}'][i] = vehicle.corners[label].steering.state.self_aligning_torque

        vx = debug_traces[f'V_wheel_x_{label}'][i]
        vy = debug_traces[f'V_wheel_y_{label}'][i]
        debug_traces[f'alpha_{label}'][i] = np.arctan2(vy, vx ) 

        corner = vehicle.corners[label]
        long = corner.longitudinal_tire
        if abs(vx) < long.params.v_min:
            debug_traces[f'kappa_{label}'][i] = 0.0
        else:
            V_wheel = omega_wheel * long.params.R_eff
            debug_traces[f'kappa_{label}'][i] = (V_wheel - vx) / abs(vx)
        
        output_traces[f'F_s_{label}'][i] = corner_state['F_s']
        output_traces[f'F_x_{label}'][i] = corner_state['F_x_tire']
        output_traces[f'F_y_{label}'][i] = corner_state['F_y_tire']
        output_traces[f'F_z_{label}'][i] = corner_state['F_z']
        
        output_traces[f'delta_s_{label}'][i] = sus_state['delta_s']
        output_traces[f'delta_t_{label}'][i] = sus_state['delta_t']
        output_traces[f'z_u_{label}'][i] = sus_state['z_u_abs']
    
    # 범프 통과 시점 상세 출력 (x = 0.5~0.7m 구간)
    if 0.49 <= vehicle.state.x <= 0.75 and i % 10 == 0:
        print(f"\n[범프 통과! t={i*dt:.3f}s, x={vehicle.state.x:.3f}m]")
        for label in ['FL', 'FR']:
            sus_state = vehicle.corners[label].suspension.get_state()
            print(f"  {label}: z_road={z_road[label]*1000:5.2f}mm, delta_t={sus_state['delta_t']*1000:6.2f}mm, F_s={sus_state['F_s']:8.1f}N, z_u={sus_state['z_u_abs']:.4f}m")
    
    # 진행 상황
    if i % 500 == 0:
        print(f"\n진행: {i}/{n_steps} ({i/n_steps*100:.1f}%)")

print("\n시뮬레이션 완료!")

# F_s 최대값 찾기
max_Fs_FL = np.max(np.abs(output_traces['F_s_FL']))
max_Fs_idx = np.argmax(np.abs(output_traces['F_s_FL']))
print(f"\nF_s 최대값: {max_Fs_FL:.0f} N at t={max_Fs_idx*dt:.3f}s, x={output_traces['x'][max_Fs_idx]:.3f}m")
print(f"  delta_t = {output_traces['delta_t_FL'][max_Fs_idx]*1000:.2f} mm")
print(f"  z_road = {input_traces['z_road_FL'][max_Fs_idx]*1000:.2f} mm")

시뮬레이션 시작...

초기 코너 상태 (디버깅):
  FL: z_u_abs=0.2993m, delta_s=-49.05mm, delta_t=16.72mm
  FR: z_u_abs=0.2993m, delta_s=-49.05mm, delta_t=16.72mm
  RR: z_u_abs=0.2993m, delta_s=-49.05mm, delta_t=16.72mm
  RL: z_u_abs=0.2993m, delta_s=-49.05mm, delta_t=16.72mm

진행: 0/200000 (0.0%)

[범프 통과! t=0.100s, x=0.507m]
  FL: z_road=29.45mm, delta_t= 16.72mm, F_s=  3188.3N, z_u=0.2993m
  FR: z_road=49.91mm, delta_t= 16.72mm, F_s=  3188.3N, z_u=0.2993m

[범프 통과! t=0.110s, x=0.557m]
  FL: z_road=29.04mm, delta_t= 16.72mm, F_s=  3188.3N, z_u=0.2993m
  FR: z_road=50.00mm, delta_t= 16.72mm, F_s=  3188.3N, z_u=0.2993m

[범프 통과! t=0.120s, x=0.608m]
  FL: z_road=28.50mm, delta_t= 16.72mm, F_s=  3188.3N, z_u=0.2993m
  FR: z_road=49.89mm, delta_t= 16.72mm, F_s=  3188.3N, z_u=0.2993m

[범프 통과! t=0.130s, x=0.658m]
  FL: z_road=27.85mm, delta_t= 16.72mm, F_s=  3188.2N, z_u=0.2993m
  FR: z_road=49.58mm, delta_t= 16.72mm, F_s=  3188.2N, z_u=0.2993m

[범프 통과! t=0.140s, x=0.708m]
  FL: z_road=27.09mm, delta_t= 16.72mm, F

## 6. 입력 신호 (Input Signals)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharex=True)
fig.suptitle('Input Signals', fontsize=14, fontweight='bold')

# 노면 프로파일
axes[0].plot(t, input_traces['z_road_FL']*1000, label='FL', linewidth=1.5)
axes[0].plot(t, input_traces['z_road_FR']*1000, label='FR', linewidth=1.5)
axes[0].plot(t, input_traces['z_road_RL']*1000, label='RL', linewidth=1.5)
axes[0].plot(t, input_traces['z_road_RR']*1000, label='RR', linewidth=1.5)
axes[0].set_ylabel('Road Height [mm]')
axes[0].set_xlabel('Time [s]')
axes[0].set_title('Road Profile')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# 드라이브 토크
axes[1].plot(t, input_traces['T_Drv'], label='T_Drv', linewidth=1.5)
axes[1].set_ylabel('Torque [Nm]')
axes[1].set_xlabel('Time [s]')
axes[1].set_title('Drive Torque Input')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. 출력 신호 (Output Signals)

### 7.1 차체 위치 및 자세

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
fig.suptitle('Vehicle Body State (Position & Attitude)', fontsize=14, fontweight='bold')

# 위치
axes[0, 0].plot(t, output_traces['x'], label='X')
axes[0, 0].set_ylabel('X Position [m]')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

axes[0, 1].plot(t, output_traces['y'], label='Y')
axes[0, 1].set_ylabel('Y Position [m]')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

axes[1, 0].plot(t, output_traces['heave']*1000, label='Heave')
axes[1, 0].set_ylabel('Heave [mm]')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

# 자세
axes[1, 1].plot(t, np.rad2deg(output_traces['roll']), label='Roll')
axes[1, 1].set_ylabel('Roll [deg]')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

axes[2, 0].plot(t, np.rad2deg(output_traces['pitch']), label='Pitch')
axes[2, 0].set_ylabel('Pitch [deg]')
axes[2, 0].set_xlabel('Time [s]')
axes[2, 0].grid(True, alpha=0.3)
axes[2, 0].legend()

axes[2, 1].plot(t, np.rad2deg(output_traces['yaw']), label='Yaw')
axes[2, 1].set_ylabel('Yaw [deg]')
axes[2, 1].set_xlabel('Time [s]')
axes[2, 1].grid(True, alpha=0.3)
axes[2, 1].legend()

plt.tight_layout()
plt.show()

### 7.2 차체 속도

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
fig.suptitle('Vehicle Body Velocities', fontsize=14, fontweight='bold')

# 선속도
axes[0, 0].plot(t, output_traces['velocity_x'], label='Velocity X')
axes[0, 0].set_ylabel('Velocity X [m/s]')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

axes[0, 1].plot(t, output_traces['velocity_y'], label='Velocity Y')
axes[0, 1].set_ylabel('Velocity Y [m/s]')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

axes[1, 0].plot(t, output_traces['heave_dot'], label='Heave Dot')
axes[1, 0].set_ylabel('Heave Dot [m/s]')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

# 각속도
axes[1, 1].plot(t, np.rad2deg(output_traces['roll_rate']), label='Roll Rate')
axes[1, 1].set_ylabel('Roll Rate [deg/s]')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

axes[2, 0].plot(t, np.rad2deg(output_traces['pitch_rate']), label='Pitch Rate')
axes[2, 0].set_ylabel('Pitch Rate [deg/s]')
axes[2, 0].set_xlabel('Time [s]')
axes[2, 0].grid(True, alpha=0.3)
axes[2, 0].legend()

axes[2, 1].plot(t, np.rad2deg(output_traces['yaw_rate']), label='Yaw Rate')
axes[2, 1].set_ylabel('Yaw Rate [deg/s]')
axes[2, 1].set_xlabel('Time [s]')
axes[2, 1].grid(True, alpha=0.3)
axes[2, 1].legend()

plt.tight_layout()
plt.show()

## 8. 검증

In [ ]:
print("=" * 50)
print("시뮬레이션 결과")
print("=" * 50)
print(f"초기 속도: {output_traces['velocity_x'][0]:.2f} m/s ({output_traces['velocity_x'][0]*3.6:.1f} km/h)")
print(f"최종 속도: {output_traces['velocity_x'][-1]:.2f} m/s ({output_traces['velocity_x'][-1]*3.6:.1f} km/h)")
print(f"\n최종 위치: x={output_traces['x'][-1]:.2f} m")
print(f"최종 자세:")
print(f"  Pitch: {np.rad2deg(output_traces['pitch'][-1]):.3f} deg")
print(f"  Roll: {np.rad2deg(output_traces['roll'][-1]):.3f} deg")
print(f"  Heave: {output_traces['heave'][-1]*1000:.2f} mm")

# 코너별 힘 플롯 추가
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
fig.suptitle('Corner Forces', fontsize=14, fontweight='bold')

# F_s (서스펜션 힘)
axes[0].plot(t, output_traces['F_s_FL'], label='FL', linewidth=1.5)
axes[0].plot(t, output_traces['F_s_FR'], label='FR', linewidth=1.5)
axes[0].plot(t, output_traces['F_s_RL'], label='RL', linewidth=1.5)
axes[0].plot(t, output_traces['F_s_RR'], label='RR', linewidth=1.5)
#axes[0].set_ylim([2500, 4000])  # y축 범위 고정

axes[0].set_ylabel('F_s [N]')
axes[0].set_title('Suspension Force (F_s)')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# F_z (타이어 수직력)
axes[1].plot(t, output_traces['F_z_FL'], label='FL', linewidth=1.5)
axes[1].plot(t, output_traces['F_z_FR'], label='FR', linewidth=1.5)
axes[1].plot(t, output_traces['F_z_RL'], label='RL', linewidth=1.5)
axes[1].plot(t, output_traces['F_z_RR'], label='RR', linewidth=1.5)
axes[1].set_ylabel('F_z [N]')
axes[1].set_title('Tire Vertical Force (F_z)')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# F_x (종방향 타이어 힘)
axes[2].plot(t, output_traces['F_x_FL'], label='FL', linewidth=1.5)
axes[2].plot(t, output_traces['F_x_FR'], label='FR', linewidth=1.5)
axes[2].plot(t, output_traces['F_x_RL'], label='RL', linewidth=1.5)
axes[2].plot(t, output_traces['F_x_RR'], label='RR', linewidth=1.5)
axes[2].set_ylabel('F_x [N]')
axes[2].set_title('Tire Longitudinal Force (F_x)')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

# F_y (횡방향 타이어 힘)
axes[3].plot(t, output_traces['F_y_FL'], label='FL', linewidth=1.5)
axes[3].plot(t, output_traces['F_y_FR'], label='FR', linewidth=1.5)
axes[3].plot(t, output_traces['F_y_RL'], label='RL', linewidth=1.5)
axes[3].plot(t, output_traces['F_y_RR'], label='RR', linewidth=1.5)
axes[3].set_ylabel('F_y [N]')
axes[3].set_xlabel('Time [s]')
axes[3].set_title('Tire Lateral Force (F_y)')
axes[3].grid(True, alpha=0.3)
axes[3].legend()

plt.tight_layout()
plt.show()

# Steering / Slip / Inputs 시각화
fig, axes = plt.subplots(5, 1, figsize=(14, 14), sharex=True)
fig.suptitle('Steering / Slip / Inputs', fontsize=14, fontweight='bold')

# 입력 토크
axes[0].plot(t, input_traces['T_steer'], label='T_steer')
axes[0].plot(t, input_traces['T_Drv'], label='T_Drv')
axes[0].plot(t, input_traces['T_brk'], label='T_brk')
axes[0].plot(t, input_traces['T_susp'], label='T_susp')
axes[0].set_ylabel('Torque [Nm]')
axes[0].set_title('Inputs')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# 조향각(steering_angle)
for label in vehicle.wheel_labels:
    axes[1].plot(t, np.rad2deg(debug_traces[f'steering_angle_{label}']), label=label, linewidth=1.5)
axes[1].set_ylabel('steering_angle [deg]')
axes[1].set_title('Steering Angle (steering_angle)')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# 슬립각(alpha)
for label in vehicle.wheel_labels:
    axes[2].plot(t, np.rad2deg(debug_traces[f'alpha_{label}']), label=label, linewidth=1.5)
axes[2].set_ylabel('alpha [deg]')
axes[2].set_title('Slip Angle (alpha)')
axes[2].grid(True, alpha=0.3)

# 슬립비(kappa)
for label in vehicle.wheel_labels:
    axes[3].plot(t, debug_traces[f'kappa_{label}'], label=label, linewidth=1.5)
axes[3].set_ylabel('kappa [-]')
axes[3].set_title('Slip Ratio (kappa)')
axes[3].grid(True, alpha=0.3)

# 휠 횡속도(V_wheel_y)
for label in vehicle.wheel_labels:
    axes[4].plot(t, debug_traces[f'V_wheel_y_{label}'], label=label, linewidth=1.5)
axes[4].set_ylabel('V_wheel_y [m/s]')
axes[4].set_xlabel('Time [s]')
axes[4].set_title('Wheel Lateral Velocity (V_wheel_y)')
axes[4].grid(True, alpha=0.3)
axes[4].legend(ncol=4)

plt.tight_layout()
plt.show()